# Simulación reproducible de manos independientes de 5 cartas

Este notebook usa el evaluador validado en `src/poker_sim/`. Cada mano se extrae de un mazo completo y es independiente de las demás.

Ejecutá todas las celdas desde el principio (`Run All`).

In [ ]:
from collections import Counter
from pathlib import Path
from random import Random
import sys

RAIZ_PROYECTO = Path.cwd()
sys.path.insert(0, str(RAIZ_PROYECTO / "src"))

from poker_sim import CATEGORIAS_EN_ORDEN, evaluar_mano, mazo_estandar

SEMILLA = 20260805
NUMERO_DE_MANOS = 100_000

## Generación

`Random` es local al experimento. Al reiniciarlo con la misma semilla se obtiene exactamente la misma muestra.

In [ ]:
mazo = mazo_estandar()
generador = Random(SEMILLA)

def generar_mano():
    """Genera una mano independiente de cinco cartas sin repetición."""
    return tuple(generador.sample(mazo, k=5))

manos = [generar_mano() for _ in range(NUMERO_DE_MANOS)]

## Evaluación y resumen

El evaluador devuelve una categoría y una puntuación de desempate. La categoría se usa aquí para contar frecuencias.

In [ ]:
evaluaciones = [evaluar_mano(mano) for mano in manos]
conteos = Counter(evaluacion.categoria for evaluacion in evaluaciones)

print(f"Semilla: {SEMILLA} | Manos simuladas: {NUMERO_DE_MANOS:,}")
print(f"Ejemplo: {' '.join(map(str, manos[0]))} ({evaluaciones[0].nombre})\n")

for categoria in CATEGORIAS_EN_ORDEN:
    cantidad = conteos[categoria]
    print(f"{categoria.name:22} {cantidad:>7,}  {cantidad / NUMERO_DE_MANOS:>8.4%}")

## Comprobaciones

Las comprobaciones confirman que la simulación respeta el mazo y que se evaluaron todas las manos. Las pruebas exhaustivas de la matemática viven en `tests/test_evaluator.py`.

In [ ]:
assert len(mazo) == 52
assert all(len(mano) == 5 and len(set(mano)) == 5 for mano in manos)
assert sum(conteos.values()) == NUMERO_DE_MANOS

print("Simulación válida.")